In [84]:
import torch

In [85]:
#Simple Attention 

inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your  (x^1)
    [0.55, 0.87, 0.66], # journey  (x^2)
    [0.57, 0.85, 0.64], # starts   (x^3)
    [0.22, 0.58, 0.33], # with    
    [0.77, 0.25, 0.10], # one     
    [0.05, 0.80, 0.55]] # step    
)


weights = torch.rand((6,6))

In [86]:
weights @ inputs

tensor([[1.6840, 2.6468, 2.1221],
        [1.5214, 1.6032, 1.2508],
        [2.0961, 2.3774, 2.3351],
        [0.7555, 1.0437, 0.7427],
        [1.4609, 2.1744, 1.8726],
        [1.8802, 2.4807, 2.4508]])

In [87]:
weights = []
for j in range(inputs.shape[0]):
    query = inputs[j]

    results = []
    for i in range(inputs.shape[0]):
        results.append(torch.dot(query,inputs[i]))

    weights.append(results)

print(torch.tensor(weights))

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [88]:
weights = inputs @ inputs.T

In [89]:
weights.sum(dim=1).shape
weights = torch.softmax(weights,dim=-1)
weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [90]:
context_vecs = weights @ inputs
print(context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [91]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad=False)

# QKV projections
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

# Compute QKV projections for all other inputs

Q = inputs @ W_query
K = inputs @ W_key
V = inputs @ W_value

attention_scores = torch.softmax((Q @ K.T)/d_out**0.5,dim=-1)
print(attention_scores)

tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])


In [92]:
context = attention_scores @ V
print(context)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


In [93]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        self.Q = x @ self.W_query
        self.K = x @ self.W_key
        self.V = x @ self.W_value

        attention_scores = torch.softmax((self.Q @ self.K.T) / self.Q.shape[-1] **0.5,dim=-1)
        context_vecs = attention_scores @ self.V
        return context_vecs
    

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [94]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in,d_out,qkv_bias=False,device = "cpu"):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias,device=device)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias,device=device)
        self.W_value =  nn.Linear(d_in,d_out,bias=qkv_bias,device=device)

    def forward(self,x):
        Q_proj = self.W_query(x)
        K_proj = self.W_key(x)
        V_proj = self.W_value(x)

        att_scores = Q_proj @ K_proj.T 
        mask = torch.triu(torch.ones_like(att_scores),diagonal=1)
        masked = att_scores.masked_fill(mask.bool(),-torch.inf)
        att_weights = torch.softmax(masked/K_proj.shape[-1]**0.5,dim=-1)
        context_vec = att_weights @ V_proj
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.4519,  0.2216],
        [-0.5874,  0.0058],
        [-0.6300, -0.0632],
        [-0.5675, -0.0843],
        [-0.5526, -0.0981],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


In [95]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)   
example = torch.ones(6, 6)     
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [96]:
class CausalAttention(nn.Module):
    def __init__(self, d_in,d_out,context_length,dropout = 0.1,qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value =  nn.Linear(d_in,d_out,bias=qkv_bias)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones((context_length,context_length)),diagonal = 1)
        )
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self,x):
        b,token_length,d_in = x.shape
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        attn_scores = Q @ K.transpose(1,2)
        attn_scores.masked_fill_(self.mask[:token_length][:token_length].bool(),-torch.inf)
        att_weights = torch.softmax(attn_scores/d_in**0.5,dim=-1)
        att_weights = self.dropout(att_weights)
        context_vecs = att_weights @ V
        return context_vecs
    
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)   

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 6, 3])
context_vecs.shape: torch.Size([2, 6, 2])


In [99]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in,d_out,context_length,dropout,num_heads):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in,d_out,context_length,dropout) for i in range(num_heads)])


    def forward(self,x):
        result = torch.cat([head(x) for head in self.heads],dim=-1)
        return result
        
    torch.manual_seed(123)

context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 1

mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.5740,  0.2216],
         [-0.7280,  0.0144],
         [-0.7740, -0.0557],
         [-0.6974, -0.0818],
         [-0.6532, -0.0964],
         [-0.6421, -0.1063]],

        [[-0.5740,  0.2216],
         [-0.7280,  0.0144],
         [-0.7740, -0.0557],
         [-0.6974, -0.0818],
         [-0.6532, -0.0964],
         [-0.6421, -0.1063]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in,d_out,context_length,dropout,num_heads,qkv_bias = False):
        super().__init__()
        assert(d_out%num_heads == 0)
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',
                             torch.triu(torch.ones(context_length,context_length),diagonal=1))
        self.out_proj = nn.Linear(d_out,d_out,bias=qkv_bias)
        

    def forward(self,x):
        b,token_length,d_in = x.shape
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        q_s = Q.view(b,token_length,self.num_heads,self.head_dim) 
        k_s = K.view(b,token_length,self.num_heads,self.head_dim)
        v_s = V.view(b,token_length,self.num_heads,self.head_dim)


        q_s = q_s.transpose(1,2)
        k_s = k_s.transpose(1,2)
        v_s = v_s.transpose(1,2)

        attention_scores = q_s @ k_s.transpose(2,3)
        mask_bool = self.mask[:token_length,:token_length].bool()
        attention_scores.masked_fill_(mask_bool,-torch.inf)

        attn_weights = torch.softmax(attention_scores/k_s.shape[-1]**0.5,dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vecs = (attn_weights @ v_s).transpose(1,2)
        context_vecs = context_vecs.contiguous().view(b,token_length,self.d_out)
        context_vecs = self.out_proj(context_vecs)
        return context_vecs

